# 01 — QQQ Prototype: V&D Cloud, Phase 1

**Project:** Volatility & Dispersion (V&D) Cloud — personal research project
**Phase:** 1 — MVP prototype on QQQ
**Data:** public daily prices via `yfinance` only

**Research question.** Can a synthetic volatility-and-dispersion state variable, mapped through
an Ichimoku-style same-date cloud, produce market regime labels that say anything honest about
forward returns — for a single, concentrated large-cap index (QQQ)?

**Data.** Public daily adjusted prices via `yfinance`.

**Verified execution.** The revised code ran on real Yahoo data on October 13, 2026 (America/Los_Angeles), under Python 3.12.1 on Windows. All 22 tests and cached-data invariant checks passed; online/offline output hashes matched. This narrative was updated afterward without changing code cells. Result tables are exported in `results/`; interactive notebook outputs remain cleared.

**What this notebook does *not* do:** create trading rules, size positions, optimize parameters,
or claim predictive power. See the closing verdict at the end.

## 1. Data Ingestion

Pull adjusted close prices for QQQ (benchmark), TQQQ (3x leveraged overlay, used only as a
downstream forward-return comparison asset), and a fixed starter universe of large-cap constituents
used to compute cross-sectional dispersion.

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt



REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").is_file())
sys.path.insert(0, str(REPO_ROOT))
import config
from src.vdcloud import (adjusted_close, price_returns, rolling_zscore, rolling_midpoint,
                         build_cloud, forward_return, purge_cross_boundary,
                         persistence_confirmed, prior_quantile_buckets, bucket_inference,
                         label_null_test, classify_zone)
from src.repro import download


print(f"pandas  : {pd.__version__}")
print(f"numpy   : {np.__version__}")
print(f"yfinance: {yf.__version__}")

In [ ]:
START_DATE       = config.START_DATE
END_DATE         = config.END_DATE
BENCHMARK_TICKER = config.BENCHMARK_TICKER
OVERLAY_TICKER   = config.OVERLAY_TICKER
Z_WINDOW         = config.Z_WINDOW
VOL_WINDOW       = config.VOL_WINDOW
SMOOTH_WINDOW    = config.SMOOTH_WINDOW
CONVERSION_WINDOW = config.CONVERSION_WINDOW
BASE_WINDOW       = config.BASE_WINDOW
SPAN_B_WINDOW     = config.SPAN_B_WINDOW
FORWARD_HORIZONS  = config.FORWARD_HORIZONS

# Fixed large-cap starter universe used only to compute cross-sectional dispersion.
# Hardcoded for MVP simplicity -- dynamic constituent tracking is out of scope for Phase 1.
STARTER_UNIVERSE = [
    "AAPL", "MSFT", "NVDA", "AMZN", "META", "GOOGL", "GOOG", "AVGO", "TSLA", "COST",
    "NFLX", "AMD", "ADBE", "CSCO", "PEP", "QCOM", "TXN", "INTU", "AMAT", "ISRG",
    "BKNG", "AMGN", "HON", "SBUX", "ADP", "MDLZ", "GILD", "LRCX", "MU", "PANW",
    "KLAC", "SNPS", "CDNS", "MAR", "ORLY", "REGN", "VRTX", "MELI", "ABNB", "CRWD",
]

print(f"Date range   : {START_DATE} to {END_DATE or 'latest'}")
print(f"Benchmark    : {BENCHMARK_TICKER}   Overlay: {OVERLAY_TICKER}")
print(f"Universe size: {len(STARTER_UNIVERSE)} tickers")
print(f"Z / vol / smooth windows: {Z_WINDOW} / {VOL_WINDOW} / {SMOOTH_WINDOW}")
print(f"Cloud windows (conversion/base/span-b): {CONVERSION_WINDOW} / {BASE_WINDOW} / {SPAN_B_WINDOW}")

In [ ]:
all_tickers = list(dict.fromkeys(STARTER_UNIVERSE + [BENCHMARK_TICKER, OVERLAY_TICKER]))

raw = download(tickers=all_tickers, start=START_DATE, end=END_DATE,
                   auto_adjust=True, progress=True)

# yfinance's column layout has changed across versions -- handle both
# (Price, Ticker) and (Ticker, Price) multi-index orderings, plus the
# single-ticker flat-column case.
prices = adjusted_close(raw, all_tickers)
close_field = "Close (auto_adjust=True)"

print(f"Prices shape: {prices.shape}   field used: {close_field}")
print(f"Date range  : {prices.index[0].date()} to {prices.index[-1].date()}")

In [ ]:
returns = price_returns(prices)

universe_cols = [t for t in STARTER_UNIVERSE if t in returns.columns]
universe_returns = returns[universe_cols].copy()
# Require at least 2 valid constituent returns per date for dispersion to be meaningful.
# Keep the date grid; the dispersion calculation masks insufficient coverage.

benchmark_returns = returns[[BENCHMARK_TICKER]].squeeze().rename(BENCHMARK_TICKER)
overlay_returns   = returns[[OVERLAY_TICKER]].squeeze().rename(OVERLAY_TICKER)

print(f"returns shape          : {returns.shape}")
print(f"universe_returns shape : {universe_returns.shape}")
print(f"First valid return date: {returns.dropna(how='all').index[0].date()}")

## 2. Dispersion and Realized-Volatility Components

Two raw building blocks feed the signal:

- **Cross-sectional dispersion** — the standard deviation, across the starter universe, of each
  day's constituent returns. High dispersion means stocks are moving apart from each other.
- **QQQ realized volatility** — a rolling 21-day standard deviation of QQQ's own daily returns,
  annualized.

Both are plausibly correlated during stress regimes (market-wide moves widen dispersion *and*
raise realized vol at the same time). Whether they carry distinct information is a question for
later validation, not something to assume here.

In [ ]:
TRADING_DAYS = 252

valid_counts = universe_returns.count(axis=1)
dispersion_daily = universe_returns.std(axis=1, skipna=True).where(valid_counts >= 2)
dispersion_ann = dispersion_daily * np.sqrt(TRADING_DAYS)

realized_vol_daily = benchmark_returns.rolling(window=VOL_WINDOW).std()
realized_vol_ann = realized_vol_daily * np.sqrt(TRADING_DAYS)

components = pd.DataFrame({
    "dispersion_daily": dispersion_daily,
    "dispersion_ann": dispersion_ann,
    "realized_vol_daily": realized_vol_daily,
    "realized_vol_ann": realized_vol_ann,
}).dropna()

corr = components["dispersion_daily"].corr(components["realized_vol_daily"])
print(f"components shape: {components.shape}")
print(f"Correlation (dispersion_daily vs realized_vol_daily): {corr:.4f}")

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(components.index, components["dispersion_ann"], color="steelblue", linewidth=1.2, label="Dispersion (ann.)")
ax.plot(components.index, components["realized_vol_ann"], color="darkorange", linewidth=1.2, alpha=0.85, label="QQQ realized vol (ann.)")
ax.set_title("V&D Cloud Components — Dispersion vs QQQ Realized Volatility")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

**Note on correlation.** Dispersion and volatility may look correlated, particularly in stress
regimes when market-wide moves simultaneously widen cross-sectional spreads and elevate realized
volatility. Whether they add distinct information to the combined signal, or simply double-count
the same regime stress, is determined by the validation steps later in this notebook — it should
not be assumed from the correlation alone.

## 3. Rolling Z-Score Normalization

Both components are put on a common, comparable scale with a rolling Z-score
(`(x - rolling_mean) / rolling_std`, window = `Z_WINDOW` = 126 trading days, ≈ 6 months).

In [ ]:
# rolling_zscore imported from src.vdcloud

dispersion_z = rolling_zscore(dispersion_daily, Z_WINDOW)
volatility_z = rolling_zscore(realized_vol_daily, Z_WINDOW)

print(f"dispersion_z valid obs: {dispersion_z.notna().sum()}  (first valid: {dispersion_z.first_valid_index().date()})")
print(f"volatility_z valid obs: {volatility_z.notna().sum()}  (first valid: {volatility_z.first_valid_index().date()})")

## 4. Additive and Penalty Signal Variants

Two combinations of the normalized components, plus two single-component controls:

- **Additive** (`0.70 * dispersion_z + 0.30 * volatility_z`) — tests whether volatility amplifies
  opportunity alongside dispersion.
- **Penalty** (`0.70 * dispersion_z - 0.30 * volatility_z`) — tests whether volatility overwhelms
  signal clarity and should reduce the score instead.
- **Dispersion-only** / **volatility-only** — single-component baselines/controls.

None of these are assumed correct going in. Each is smoothed with a `SMOOTH_WINDOW` = 5-day
moving average before being fed into the cloud geometry below.

In [ ]:
vd_additive_raw        = 0.70 * dispersion_z + 0.30 * volatility_z
vd_penalty_raw         = 0.70 * dispersion_z - 0.30 * volatility_z
vd_dispersion_only_raw = dispersion_z.copy()
vd_volatility_only_raw = volatility_z.copy()

vd_additive        = vd_additive_raw.rolling(window=SMOOTH_WINDOW).mean()
vd_penalty         = vd_penalty_raw.rolling(window=SMOOTH_WINDOW).mean()
vd_dispersion_only = vd_dispersion_only_raw.rolling(window=SMOOTH_WINDOW).mean()
vd_volatility_only = vd_volatility_only_raw.rolling(window=SMOOTH_WINDOW).mean()

signal_variants = {
    "additive": vd_additive,
    "penalty": vd_penalty,
    "dispersion_only": vd_dispersion_only,
    "volatility_only": vd_volatility_only,
}
for name, s in signal_variants.items():
    print(f"  {name:18s} valid obs: {s.notna().sum()}")

## 5. Same-Date Cloud Geometry and Regime Labels

An Ichimoku-style cloud, applied to the *signal* (not price): a fast rolling midpoint
(conversion line, 9-day), a medium rolling midpoint (base line, 26-day), and a slow rolling
midpoint (Span B, 52-day). Span A is the average of the conversion and base lines. The cloud band
is the row-wise max/min of Span A and Span B — all computed **same-date**, with no forward
shifting.

Each date is classified `above_cloud`, `inside_cloud`, or `below_cloud` depending on where the
signal sits relative to the band. This is built for both the additive and penalty lenses, since
the later validation steps compare the two.

In [ ]:
# rolling_midpoint imported from src.vdcloud

def build_variant_cloud(signal, variant_name):
    return build_cloud(signal, CONVERSION_WINDOW, BASE_WINDOW, SPAN_B_WINDOW).dropna(subset=["regime_state"])


In [ ]:
additive_cloud = build_variant_cloud(vd_additive, "additive")
penalty_cloud  = build_variant_cloud(vd_penalty, "penalty")

for name, frame in [("additive", additive_cloud), ("penalty", penalty_cloud)]:
    print(f"--- {name} regime distribution ---")
    print(frame["regime_state"].value_counts())
    print()

fig, ax = plt.subplots(figsize=(13, 5.5))
idx = additive_cloud.index
ax.fill_between(idx, additive_cloud["cloud_bottom"], additive_cloud["cloud_top"],
                 color="lightblue", alpha=0.35, label="Cloud (Span A / Span B)")
ax.plot(idx, additive_cloud["signal"], color="steelblue", linewidth=1.4, label="vd_additive")
ax.axhline(0, color="gray", linewidth=0.8, linestyle="--", alpha=0.6)
ax.set_title("V&D Cloud — Additive Signal with Same-Date Cloud (QQQ, Phase 1)")
ax.legend(loc="upper left"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 6. Forward-Return-by-Regime Analysis

For each regime label (`above_cloud`, `inside_cloud`, `below_cloud`), summarize QQQ and TQQQ
forward returns at 5/10/21/63-day horizons. Forward return at date *t* is the compounded return
from *t+1* through *t+h* — the same-day return at *t* is excluded so the label isn't peeking at
its own outcome.

TQQQ is a 3x daily-reset leveraged product with volatility decay; its forward returns reflect that
structural behavior and should not be read as a clean read on signal quality by themselves.

In [ ]:
def calculate_forward_returns(return_series, horizons):
    return pd.DataFrame({f"fwd_{h}d": forward_return(return_series, h) for h in horizons})

def summarize_forward_returns_by_regime(validation_frame, asset_label, horizons):
    rows = []
    for h in horizons:
        col = f"fwd_{h}d"
        for regime, group in validation_frame.groupby("regime_state"):
            r = group[col].dropna()
            if r.empty:
                continue
            rows.append({
                "asset": asset_label, "horizon": h, "regime_state": regime,
                "sample_count": len(r), "mean_return": r.mean(), "median_return": r.median(),
                "win_rate": (r > 0).mean(), "std_return": r.std(),
            })
    return pd.DataFrame(rows)

qqq_fwd  = calculate_forward_returns(benchmark_returns, FORWARD_HORIZONS)
tqqq_fwd = calculate_forward_returns(overlay_returns, FORWARD_HORIZONS)

print(f"qqq_fwd shape : {qqq_fwd.shape}")
print(f"tqqq_fwd shape: {tqqq_fwd.shape}")

In [ ]:
validation_rows = []
for lens_name, cloud_frame in [("additive", additive_cloud), ("penalty", penalty_cloud)]:
    regime_col = cloud_frame[["regime_state"]]
    qqq_val  = regime_col.reindex(qqq_fwd.index).join(qqq_fwd, how="left")
    tqqq_val = regime_col.join(tqqq_fwd, how="left")
    qqq_summary  = summarize_forward_returns_by_regime(qqq_val, "QQQ", FORWARD_HORIZONS)
    tqqq_summary = summarize_forward_returns_by_regime(tqqq_val, "TQQQ", FORWARD_HORIZONS)
    qqq_summary["lens"] = lens_name
    tqqq_summary["lens"] = lens_name
    validation_rows += [qqq_summary, tqqq_summary]

validation_summary = pd.concat(validation_rows, ignore_index=True).sort_values(
    ["lens", "asset", "horizon", "regime_state"]).reset_index(drop=True)

validation_summary

## 7. Regime Persistence and Stability

Do regimes behave like durable market "weather states," or do they flicker day to day? For each
lens, compute contiguous regime runs and summarize their duration. A regime that changes every
other day is much harder to trust diagnostically than one that holds for weeks.

In [ ]:
def compute_regime_runs(regime_series, regime_name):
    regime_series = regime_series.reindex(prices.index)
    run_id = regime_series.ne(regime_series.shift()).cumsum()
    runs = (
        pd.DataFrame({"date": regime_series.index, "regime_state": regime_series.values, "run_id": run_id.values})
        .groupby("run_id")
        .agg(regime_state=("regime_state", "first"), start_date=("date", "first"),
             end_date=("date", "last"), duration_days=("date", "count"))
        .reset_index(drop=True)
    )
    runs = runs[runs["regime_state"].notna()].copy()
    runs["regime_name"] = regime_name
    return runs

all_runs = pd.concat([
    compute_regime_runs(additive_cloud["regime_state"], "additive"),
    compute_regime_runs(penalty_cloud["regime_state"], "penalty"),
], ignore_index=True)

stability_summary = (
    all_runs.groupby(["regime_name", "regime_state"])["duration_days"]
    .agg(run_count="count", mean_duration="mean", median_duration="median")
    .reset_index()
)
stability_summary

## 8. Exploratory Robustness Checks

The HAC regression uses the full chronological sample, with a bucket indicator and intercept. Its coefficient contrasts bucket and complement means; the descriptive excess column instead compares the bucket with the unconditional mean. The 21-day check uses 21 HAC lags.

The shuffled-label null keeps group counts but destroys regime persistence. A circular-shift sensitivity check preserves most runs, introduces a wrap boundary, and assumes approximate stationarity. Both use 1,000 seeded draws and plus-one Monte Carlo p-values. These complementary checks do not correct selection or multiplicity and cannot establish predictive validity.


In [ ]:
import math

try:
    import statsmodels.api as sm
    STATSMODELS_AVAILABLE = True
except ImportError:
    STATSMODELS_AVAILABLE = False

print("statsmodels available:", STATSMODELS_AVAILABLE)


# HAC uses a full chronological bucket regression below.


def summarize_regime_inference(frame, return_col, hac_lags=21):
    uncond_mean = frame.loc[frame["regime_state"].notna(), return_col].dropna().mean()
    rows = []
    for state, group in frame.dropna(subset=["regime_state", return_col]).groupby("regime_state"):
        r = group[return_col]
        hac = bucket_inference(frame, "regime_state", return_col, state, hac_lags)
        rows.append({"regime_state": state, "sample_count": len(r), "mean_return": r.mean(),
                     "excess_vs_unconditional": r.mean() - uncond_mean, "hac_t_stat": hac["hac_t_stat"]})
    return pd.DataFrame(rows)

print("HAC inference helpers defined.")

In [ ]:
hac_rows = []
for lens_name, cloud_frame in [("additive", additive_cloud), ("penalty", penalty_cloud)]:
    joined = cloud_frame[["regime_state"]].reindex(qqq_fwd.index).join(qqq_fwd[["fwd_21d"]], how="left")
    summary = summarize_regime_inference(joined, return_col="fwd_21d", hac_lags=21)
    summary["lens"] = lens_name
    hac_rows.append(summary)

hac_summary = pd.concat(hac_rows, ignore_index=True)
hac_summary

In [ ]:
def run_regime_label_permutation_test(frame, return_col, n_permutations=1000, random_seed=42):
    rows = []
    for state in frame["regime_state"].dropna().unique():
        shuffle = label_null_test(frame, "regime_state", return_col, state, n_permutations, random_seed)
        circular = label_null_test(frame, "regime_state", return_col, state, n_permutations, random_seed,
                                  method="circular_shift", min_shift=21)
        rows.append({"regime_state": state, "observed_excess": shuffle["observed_excess"],
                     "empirical_p_two_sided": shuffle["empirical_two_sided_p_value"],
                     "circular_shift_p": circular["empirical_two_sided_p_value"]})
    return pd.DataFrame(rows)

print("Permutation test helper defined.")

In [ ]:
perm_rows = []
for lens_name, cloud_frame in [("additive", additive_cloud), ("penalty", penalty_cloud)]:
    joined = cloud_frame[["regime_state"]].reindex(qqq_fwd.index).join(qqq_fwd[["fwd_21d"]], how="left")
    result = run_regime_label_permutation_test(joined, return_col="fwd_21d", n_permutations=1000, random_seed=42)
    result["lens"] = lens_name
    perm_rows.append(result)

permutation_summary = pd.concat(perm_rows, ignore_index=True)
permutation_summary

## Phase 1 Verdict — Verified Revised Run

The repaired 21-day HAC diagnostic has two heuristic |t| ≥ 1.65 crossings among six lens/regime cells. None crosses the circular-shift p ≤ 0.10 sensitivity threshold; the smallest circular-shift p is 0.103896 for penalty / below_cloud. The weak threshold evidence does not establish predictive validity.

The first valid additive and penalty labels occur on 2015-10-20, after both cloud spans exist. Missing-price returns are not filled. These invariants establish calculation behavior, not a forecasting claim. See the exported Phase 1 tables and the verified manifest in `results/`.
